# BQuant: S&P 500 option-price Z-score dashboard

**Starter notebook — initially runs on clearly labelled synthetic data. No Bloomberg prices are included.**

This notebook provides the dashboard and rolling calculations. The Bloomberg option-history source still needs to be connected and verified in your Terminal. The BQL adapter does not guess field names, reconstruct expired option chains, or create a constant-maturity surface.

The screen has a ranking table, call/put/spread Z-scores, a raw-premium versus premium/spot switch, ticker search, a minimum-|Z| filter, stock-history charts, and a manual data-refresh button. Dependencies: `numpy`, `pandas`, `matplotlib`, `ipywidgets`, and `IPython`; BQL mode additionally needs Bloomberg's `bql` package and your data entitlements.

The calculation engine was tested against synthetic data and a direct manual calculation. Widget rendering and live Bloomberg requests have **not** been tested in this environment.


## Definition and data requirements

For each stock independently, the default metrics are **call premium**, **put premium**, and **call minus put premium**. The optional normalized basis divides each premium by that day's underlying spot and expresses it as a percentage. Raw prices remain visible in USD per underlying share.

At date $t$, the baseline is the prior **365 calendar days**, $[t-365\mathrm{D},t)$, excluding today's observation. The Z-score is $(x_t-\mu)/s$ with sample standard deviation (`ddof=1`). This is a 365-day convention, not an exact anniversary window in leap years. A result needs at least **200 valid prior observations** and a full 365 days of elapsed history for that metric. Missing quotes and zero variance produce no score; the code never forward-fills them.

A sensible starting option specification is a **30-calendar-day target and at-the-money strike rule**. Apply the same specification at every historical date. A nearest-listed-expiry series is only a proxy, not an exact constant-maturity series. The matched call and put must have the same expiry and strike on each date. Verify quote timing, currency, adjusted-contract treatment, and selection rules upstream. An exactly standardized series may need historical chains or a supported surface dataset; access is not assumed here.

Do not take today's near-expiry contract and treat its older, longer-dated quotes as a constant-tenor series. Option prices also reflect spot, strike, time, volatility, rates, and dividends. A high Z-score measures unusualness, not overvaluation or a trading instruction; a call-minus-put spread is also influenced by put-call parity.

For the full last-year Z-score chart, supply about **two years of daily history**. One year supplies the baseline for the latest observation but not a full year of already-warmed-up Z-scores.


## 1. Settings

In [ ]:
# Run all cells in order. DEMO is safe and requires no Bloomberg queries.
MODE = "DEMO"                 # "DEMO", "CSV", or "BQL"
CSV_PATH = "option_history.csv"
WINDOW_DAYS = 365
MIN_OBSERVATIONS = 200

# BQL mode ONLY: replace None with verified expression-building functions.
# Each function takes (bq, dates) and returns ONE daily standardized premium
# series PER UNDERLYING. Do not supply IV fields or a current option ID's
# unadjusted history. Do not guess field names.
BQL_CALL_ITEM = None
BQL_PUT_ITEM = None

# Replace this label when using real input data; describe the actual series.
REAL_SERIES_LABEL = "REPLACE: option tenor, strike rule, quote basis, and source timing"


## 2. Validation and rolling calculations

In [ ]:
"""BQuant options dashboard starter.

This file does not contain Bloomberg market data. The demo is synthetic.
The BQL adapter requires user-verified standardized option-price expressions.
No option-history field names are guessed. It has not been executed against BQL.
"""
from __future__ import annotations

from functools import reduce
from html import escape
from typing import Callable, Optional

import numpy as np
import pandas as pd

REQUIRED = ["date", "ticker", "spot", "call_mid", "put_mid"]
METRICS = {"Call Z": "call", "Put Z": "put", "Spread Z": "spread"}


def validate_history(raw: pd.DataFrame) -> pd.DataFrame:
    """Validate one daily observation per underlying and a fixed option rule.

    Prices must be USD per underlying share; dates must be timezone-naive US
    market-date labels. Missing observations are allowed but never filled.
    The caller must verify comparable tenor/moneyness and price conventions;
    the five-column schema alone cannot establish these properties.
    """
    if not isinstance(raw, pd.DataFrame) or raw.empty:
        raise ValueError("The history loader must return a nonempty DataFrame.")
    missing = set(REQUIRED).difference(raw.columns)
    if missing:
        raise ValueError("Missing input columns: " + ", ".join(sorted(missing)))
    d = raw[REQUIRED].copy()
    d["date"] = pd.to_datetime(d["date"], errors="raise")
    if d["date"].isna().any():
        raise ValueError("Dates cannot be missing.")
    if d["date"].dt.tz is not None:
        raise ValueError("Use timezone-naive US market-date labels, not timestamps.")
    d["date"] = d["date"].dt.normalize()
    if d["ticker"].isna().any():
        raise ValueError("Ticker values cannot be missing.")
    d["ticker"] = d["ticker"].astype(str).str.strip()
    if d["ticker"].eq("").any():
        raise ValueError("Ticker values cannot be blank.")
    if d.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate date/ticker rows: select a single consistent option series first.")
    for col in ["spot", "call_mid", "put_mid"]:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(float)
        if np.isinf(d[col]).any():
            raise ValueError(f"Infinite values in {col}.")
        invalid = d[col].le(0) if col == "spot" else d[col].lt(0)
        if invalid.any():
            raise ValueError(f"Invalid negative/zero price in {col}.")
    return d.sort_values(["date", "ticker"]).reset_index(drop=True)


def compute_dashboard(
    raw: pd.DataFrame,
    basis: str = "Raw premium",
    window_days: int = 365,
    min_obs: int = 200,
    asof: Optional[str] = None,
) -> dict:
    """Calculate per-stock trailing z-scores, never cross-sectional z-scores.

    Baseline at t: dates in [t-window_days, t); today's value is excluded.
    A score requires min_obs nonmissing prior observations AND at least
    window_days of elapsed history for that stock/metric. Standard deviation
    uses ddof=1. A zero or invalid standard deviation produces NaN, not zero.
    This is a 365-day convention, not an exact anniversary in leap years.
    """
    if window_days < 2 or min_obs < 2:
        raise ValueError("window_days and min_obs must both be >= 2.")
    if basis not in {"Raw premium", "Premium / spot"}:
        raise ValueError("Unknown price basis.")
    d = validate_history(raw)
    stamp = pd.Timestamp(asof).normalize() if asof else d["date"].max()
    if stamp.tzinfo is not None:
        raise ValueError("asof must be a timezone-naive market date.")
    d = d.loc[d["date"].le(stamp)].copy()
    if d.empty:
        raise ValueError("There are no observations on or before the requested date.")
    tickers = sorted(d["ticker"].unique())
    dates = pd.DatetimeIndex(sorted(set(d["date"]) | {stamp}), name="date")
    price = {
        col: d.pivot(index="date", columns="ticker", values=col).reindex(
            index=dates, columns=tickers
        )
        for col in ["spot", "call_mid", "put_mid"]
    }
    call, put = price["call_mid"], price["put_mid"]
    if basis == "Premium / spot":
        call = 100.0 * call.div(price["spot"])
        put = 100.0 * put.div(price["spot"])
    values = {"call": call, "put": put, "spread": call - put}
    scores, counts, means, stds, statuses = {}, {}, {}, {}, {}
    delta = pd.Timedelta(days=window_days)
    for metric, x in values.items():
        rolling = x.rolling(f"{window_days}D", closed="left", min_periods=2)
        mean, std = rolling.mean(), rolling.std(ddof=1)
        count = x.rolling(f"{window_days}D", closed="left", min_periods=0).count()
        mature = pd.DataFrame(False, index=dates, columns=tickers)
        for ticker in tickers:
            first = x[ticker].first_valid_index()
            if first is not None:
                mature[ticker] = dates >= first + delta
        eligible = count.ge(min_obs) & mature
        z = ((x - mean) / std.where(std.gt(1e-12))).where(eligible)
        scores[metric], counts[metric] = z, count
        means[metric], stds[metric] = mean, std
        status = pd.Series("OK", index=tickers, dtype=object)
        status.loc[~std.loc[stamp].gt(1e-12)] = "Zero/invalid standard deviation"
        status.loc[count.loc[stamp].lt(min_obs)] = f"Fewer than {min_obs} prior observations"
        status.loc[~mature.loc[stamp]] = f"Less than {window_days} days of history"
        status.loc[x.loc[stamp].isna()] = "No value at the as-of date"
        statuses[metric] = status

    latest = pd.DataFrame(index=pd.Index(tickers, name="Ticker"))
    latest["Spot"] = price["spot"].loc[stamp]
    latest["Call mid"] = price["call_mid"].loc[stamp]
    latest["Put mid"] = price["put_mid"].loc[stamp]
    for label, metric in METRICS.items():
        latest[label] = scores[metric].loc[stamp]
    quote_rows = d.loc[d[["call_mid", "put_mid"]].notna().any(axis=1)]
    last_quote = quote_rows.groupby("ticker")["date"].max()
    latest["Last option date"] = last_quote.reindex(tickers)
    return dict(
        latest=latest, scores=scores, values=values, price=price,
        counts=counts, means=means, stds=stds, statuses=statuses,
        asof=stamp, window_days=window_days, min_obs=min_obs, basis=basis,
    )



## 3. Demo data and BQL adapter

In [ ]:

def make_demo_history(end: Optional[str] = None, seed: int = 47) -> pd.DataFrame:
    """Fictitious data for UI/calculation testing; NOT quotes or a pricing model."""
    rng = np.random.default_rng(seed)
    end_day = (
        pd.Timestamp(end) if end else
        pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
        - pd.Timedelta(days=1)
    )
    dates = pd.bdate_range(end=end_day, periods=570)
    pieces = []
    for j in range(12):
        spot = (50 + 15 * j) * np.exp(np.cumsum(rng.normal(0, 0.009, len(dates))))
        common = 0.025 + 0.004 * np.sin(np.arange(len(dates)) / (23 + j))
        call = spot * np.maximum(0.001, common + rng.normal(0, 0.0015, len(dates)))
        put = spot * np.maximum(0.001, common + rng.normal(0, 0.0015, len(dates)))
        if j == 0:
            call[-1] *= 1.55
        if j == 1:
            put[-1] *= 1.60
        if j == 9:
            call[:430] = np.nan
            put[:430] = np.nan
        if j == 10:
            call[-1] = np.nan
            put[-1] = np.nan
        if j == 11:
            call[:] = 3.0
            put[:] = 2.0
        pieces.append(pd.DataFrame(dict(
            date=dates, ticker=f"DEMO{j+1:02d}", spot=spot,
            call_mid=call, put_mid=put,
        )))
    return pd.concat(pieces, ignore_index=True)


def fetch_bql_standardized_history(
    call_item: Callable,
    put_item: Callable,
    start: Optional[str] = None,
    end: Optional[str] = None,
    batch_size: int = 40,
) -> pd.DataFrame:
    """BQL adapter pattern; must be validated in the user's Bloomberg session.

    call_item(bq, dates) / put_item(bq, dates) must return USER-VERIFIED BQL
    expressions that produce standardized daily option premiums per underlying
    (e.g. a consistently constructed 30-day ATM series), NOT IV values and NOT
    today's selected contract backfilled over history. Public sources did not
    establish the needed field/entitlement availability. This function does NOT
    build a constant-maturity surface or historical option chain.

    Default end is yesterday in New York: this adapter is daily/EOD, not a
    live streaming subscription. Check source field timing and fill defaults.
    All three fields must have compatible EOD dating and pricing conventions.
    Current SPX members are used; this is not a point-in-time index backtest.
    """
    if not callable(call_item) or not callable(put_item):
        raise ValueError("Supply verified call_item and put_item BQL expression builders.")
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    try:
        import bql
    except ImportError as exc:
        raise ImportError("Run the BQL adapter inside an entitled Bloomberg BQuant session.") from exc
    bq = bql.Service()
    end_day = (
        pd.Timestamp(end).normalize() if end else
        pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
        - pd.Timedelta(days=1)
    )
    start_day = pd.Timestamp(start) if start else end_day - pd.DateOffset(years=2, days=10)
    if start_day > end_day:
        raise ValueError("start must not be later than end.")
    membership = bq.execute(bql.Request(
        bq.univ.members("SPX Index"), {"Name": bq.data.name()}
    ))[0].df().reset_index()
    if "ID" not in membership:
        raise ValueError("Unexpected BQL membership response: expected an ID column.")
    universe = membership["ID"].dropna().astype(str).drop_duplicates().tolist()
    if not universe:
        raise ValueError("BQL returned no current SPX Index members.")
    dates = bq.func.range(start_day.strftime("%Y-%m-%d"), end_day.strftime("%Y-%m-%d"))
    results = []
    for offset in range(0, len(universe), batch_size):
        items = dict(
            spot=bq.data.px_last(dates=dates),
            call_mid=call_item(bq, dates),
            put_mid=put_item(bq, dates),
        )
        response = bq.execute(bql.Request(universe[offset:offset + batch_size], items))
        if len(response) != len(items):
            raise ValueError("BQL did not return the three requested history items.")
        frames = []
        for name, item_response in zip(items, response):
            frame = item_response.df().reset_index()
            needed = {"ID", "DATE", name}
            if not needed.issubset(frame.columns):
                raise ValueError(f"Unexpected columns for {name}: {list(frame.columns)}")
            frame = frame[["ID", "DATE", name]].rename(columns={"ID": "ticker", "DATE": "date"})
            frame["date"] = pd.to_datetime(frame["date"])
            if frame.duplicated(["ticker", "date"]).any():
                raise ValueError(f"{name} has multiple rows per stock/date; fix tenor/moneyness dimensions.")
            frames.append(frame)
        results.append(reduce(
            lambda left, right: left.merge(right, on=["ticker", "date"], how="outer", validate="one_to_one"),
            frames,
        ))
    panel = pd.concat(results, ignore_index=True)
    panel = panel.loc[panel[["spot", "call_mid", "put_mid"]].notna().any(axis=1)].copy()
    if panel.empty:
        raise ValueError("BQL returned no nonmissing observations.")
    absent = sorted(set(universe) - set(panel["ticker"]))
    if absent:
        placeholder = pd.DataFrame({"ticker": absent, "date": panel["date"].max()})
        for name in ["spot", "call_mid", "put_mid"]:
            placeholder[name] = np.nan
        panel = pd.concat([panel, placeholder], ignore_index=True)
    return validate_history(panel)



## 4. Dashboard interface

In [ ]:

def launch_dashboard(
    loader: Callable[[], pd.DataFrame],
    source_label: str = "DEMO — synthetic data, not Bloomberg prices",
    series_label: str = "Verify a consistent option tenor and moneyness in the data loader",
    window_days: int = 365,
    min_obs: int = 200,
):
    """Build a notebook widget dashboard. Refresh invokes loader; filters do not."""
    import ipywidgets as w
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display

    basis = w.Dropdown(options=["Raw premium", "Premium / spot"], description="Basis:")
    metric = w.Dropdown(options=list(METRICS), value="Spread Z", description="Rank by:")
    minimum = w.FloatSlider(value=0, min=0, max=4, step=0.5, description="Min |Z|:", continuous_update=False)
    top_n = w.Dropdown(options=[20, 50, 100, 600], value=50, description="Max rows:")
    search = w.Text(placeholder="Ticker contains…", description="Search:", continuous_update=False)
    stock = w.Dropdown(options=[], description="Chart:", layout=w.Layout(width="330px"))
    refresh = w.Button(description="Refresh data", icon="refresh")
    banner = w.HTML()
    table_out, chart_out = w.Output(), w.Output()
    state = {"raw": None, "models": {}, "loading": False}

    def model():
        if basis.value not in state["models"]:
            state["models"][basis.value] = compute_dashboard(
                state["raw"], basis.value, window_days=window_days, min_obs=min_obs
            )
        return state["models"][basis.value]

    def draw_table(*_):
        if state["loading"] or state["raw"] is None:
            return
        m = model()
        key = METRICS[metric.value]
        all_rows = m["latest"].copy()
        all_rows["Prior observations"] = m["counts"][key].loc[m["asof"]].astype(int)
        all_rows["Status"] = m["statuses"][key]
        valid = all_rows[metric.value].notna()
        t = all_rows.copy()
        if search.value.strip():
            t = t.loc[t.index.str.contains(search.value.strip(), case=False, regex=False)]
        if minimum.value > 0:
            t = t.loc[t[metric.value].abs().ge(minimum.value)]
        t = t.assign(_rank=t[metric.value].abs()).sort_values("_rank", ascending=False, na_position="last")
        t = t.drop(columns="_rank").head(top_n.value)
        t["Last option date"] = t["Last option date"].dt.strftime("%Y-%m-%d").fillna("—")
        with table_out:
            table_out.clear_output(wait=True)
            display(HTML(
                f"<p><b>{int(valid.sum())}/{len(all_rows)} stocks have a valid {escape(metric.value)}.</b> "
                f"As of input date {m['asof']:%Y-%m-%d}. "
                "Rows without a usable latest observation are not carried forward.</p>"
            ))
            if t.empty:
                display(HTML("<p>No rows match these filters.</p>"))
            else:
                display(HTML(t.to_html(float_format=lambda v: f"{v:,.2f}", na_rep="—", escape=True)))

    def draw_chart(*_):
        if state["loading"] or state["raw"] is None or not stock.value:
            return
        m, ticker = model(), stock.value
        left = m["asof"] - pd.Timedelta(days=365)
        with chart_out:
            chart_out.clear_output(wait=True)
            fig = plt.figure(figsize=(11, 3.4))
            for label, key in METRICS.items():
                z = m["scores"][key].loc[left:, ticker]
                plt.plot(z.index, z.values, label=label, linewidth=1.25)
            plt.axhline(2, linestyle="--", linewidth=0.8, label="±2 reference")
            plt.axhline(-2, linestyle="--", linewidth=0.8)
            plt.axhline(0, linestyle=":", linewidth=0.8)
            plt.title(f"{ticker} | Trailing {window_days}-day Z | {basis.value}")
            plt.ylabel("Standard deviations")
            plt.legend(loc="best")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            fig = plt.figure(figsize=(11, 3.0))
            for label, key in [("Call mid", "call_mid"), ("Put mid", "put_mid")]:
                p = m["price"][key].loc[left:, ticker]
                plt.plot(p.index, p.values, label=label, linewidth=1.25)
            plt.title(f"{ticker} | Input option premiums")
            plt.ylabel("USD per underlying share")
            plt.legend(loc="best")
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def refresh_data(*_):
        refresh.disabled = True
        state["loading"] = True
        try:
            raw = validate_history(loader())
            state["raw"], state["models"] = raw, {}
            tickers = sorted(raw["ticker"].unique())
            old = stock.value
            stock.options = tickers
            stock.value = old if old in tickers else tickers[0]
            loaded = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
            banner.value = (
                f"<h3>S&amp;P 500 options — rolling Z-score dashboard</h3>"
                f"<p><b>{escape(source_label)}</b><br>{escape(series_label)}<br>"
                f"Latest input date: {raw['date'].max():%Y-%m-%d}; loaded {loaded}. "
                "Daily observations; not a live streaming feed.</p>"
                f"<p>Baseline: prior {window_days} calendar days, excluding the current date; "
                f"minimum {min_obs} valid prior observations and a full elapsed window. "
                "Prices are always shown in USD/share; Z-scores follow the selected basis. "
                "Z-scores are unusualness measures, not buy/sell signals.</p>"
            )
        except Exception as exc:
            state["raw"], state["models"] = None, {}
            banner.value = f"<b>Data loading failed:</b> {escape(str(exc))}"
            table_out.clear_output()
            chart_out.clear_output()
        finally:
            state["loading"] = False
            refresh.disabled = False
        if state["raw"] is not None:
            draw_table()
            draw_chart()

    for control in [metric, minimum, top_n, search]:
        control.observe(draw_table, names="value")
    stock.observe(draw_chart, names="value")
    basis.observe(lambda *_: (draw_table(), draw_chart()), names="value")
    refresh.on_click(refresh_data)
    controls = w.Box(
        [basis, metric, minimum, top_n, search, refresh],
        layout=w.Layout(display="flex", flex_flow="row wrap"),
    )
    ui = w.VBox([banner, controls, table_out, stock, chart_out])
    refresh_data()
    return ui


## Connect the input data

**DEMO:** Runs immediately when notebook dependencies are present. Fictitious names such as `DEMO01` are not S&P 500 constituents, and the generated premiums are not market data or a financial pricing model.

**CSV:** Set `MODE = "CSV"` and provide the five columns below. Only use current S&P 500 members in this file, or explicitly filter your existing dataset before passing it to the dashboard. The CSV path does not independently check index membership.

| Column | Required meaning |
|---|---|
| `date` | US market date, one observation per underlying/date, such as `2026-09-28` |
| `ticker` | Consistent Bloomberg underlying identifier |
| `spot` | Compatible daily underlying price in USD/share |
| `call_mid` | Call premium in USD per underlying share, consistently selected through time |
| `put_mid` | Matched put premium on the same price and selection basis |

Use `NaN` for missing observations. Do not substitute volume, open interest, or implied volatility for an option premium. If a source produces synthetic/model-derived standardized premiums rather than bid/ask midpoints, make that explicit in the source label and adjust column labels before production use.

**BQL:** The adapter below illustrates membership retrieval for `SPX Index`, batching, and history alignment. You must supply `BQL_CALL_ITEM` and `BQL_PUT_ITEM` using expressions verified in your BQuant field documentation. Their field availability, time-series parameters, licensing, and timing have not been established here. They must produce one value per underlying/date, not an unfiltered option chain. If no suitable standardized price fields exist, build the history from archived chains outside this adapter and use CSV or your own loader.

The default BQL end date is yesterday in New York and the result is a **latest available daily/EOD snapshot**, not a live streaming quote feed. Confirm the data items' own source timestamps, fill defaults, and EOD conventions. The Python code adds no forward filling, but it cannot detect values already filled by the upstream source.

For a live snapshot version, fetch a compatible synchronized call/put/spot snapshot separately, replace that date's row, and keep the current day out of the historical baseline. The present notebook does not subscribe to live data.

Current S&P 500 membership is suitable for a current-universe screen. A historical index backtest needs membership as of each historical date; that is not implemented here.


## 5. Launch

In [ ]:
from IPython.display import display

def load_history():
    if MODE == "DEMO":
        return make_demo_history()
    if MODE == "CSV":
        return pd.read_csv(CSV_PATH)
    if MODE == "BQL":
        if BQL_CALL_ITEM is None or BQL_PUT_ITEM is None:
            raise ValueError(
                "BQL option-history expressions are not configured. "
                "Verify your standardized call and put premium data items in BQuant, "
                "then assign BQL_CALL_ITEM and BQL_PUT_ITEM."
            )
        return fetch_bql_standardized_history(BQL_CALL_ITEM, BQL_PUT_ITEM)
    raise ValueError("MODE must be DEMO, CSV, or BQL.")

is_demo = MODE == "DEMO"
dashboard = launch_dashboard(
    load_history,
    source_label=(
        "DEMO — synthetic data only; not Bloomberg prices or actual S&P 500 members"
        if is_demo else f"{MODE} data — check latest input date and source timing"
    ),
    series_label=("Fictitious premium series for testing only" if is_demo else REAL_SERIES_LABEL),
    window_days=WINDOW_DAYS,
    min_obs=MIN_OBSERVATIONS,
)
display(dashboard)


## Primary-source references

Bloomberg BQuant product overview: `https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/`

pandas rolling-window semantics: `https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html`

Jupyter widget events: `https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Events.html`

Options Industry Council, option-pricing factors: `https://www.optionseducation.org/optionsoverview/options-pricing`

Options Industry Council, put/call parity: `https://www.optionseducation.org/advancedconcepts/put-call-parity`

These references support the platform, calculation mechanics, and option-pricing cautions. They do **not** verify the Bloomberg standardized option-price fields needed by the data adapter.
